In [42]:

import ase
from ase.cluster.cubic import FaceCenteredCubic
from ase.visualize import view
from ase.io import write

surfaces = [(1,0,0),(1,1,0), (1,1,1)]
layers = [3,4,3]
lc = 3.61
qd = FaceCenteredCubic('Cu', surfaces, layers, latticeconstant=lc)

qd.center(vacuum=10)
print(len(qd))
# Optionally turn on PBCs if you want to simulate in periodic DFT
qd.pbc = [True, True, True]
view(qd, viewer= 'x3d')

# write("/home/fes33/Documents/Temp/QD/Cu_cluster.cif", qd)

87


In [47]:
from ase.build import bulk, surface
from ase.visualize import view
from ase.io import write

# Step 1: Create bulk ZrO2
a = 5.09
bulk_zro2 = bulk('ZrO2', 'fluorite', a=a)
layer_separation = 2
slab_vacuum, axis = 10, 2
miller = (1,1,1)
supercell_size = (8,8,3)



# Step 2: Make (111) slab
slab = surface(bulk_zro2, miller, 1, vacuum=layer_separation)

# Step 3: Supercell replication
supercell = slab * supercell_size
supercell.pbc = [True, True, False]
supercell.center(vacuum=slab_vacuum, axis=axis)

# Step 4: Visualize and export
view(supercell, viewer='x3d')
# write("ZrO2_slab_supercell.cif", supercell)


In [50]:
from ase import Atoms
from ase.build import add_adsorbate
from ase.visualize import view


slab_supercell = supercell
# Step 2: Find slab height (zmax)
zmax_slab = slab_supercell.positions[:, 2].max()

# Step 3: Shift QD above the slab by some distance
vacuum_gap = 3.0  # Å between slab surface and QD bottom
zmin_qd = qd.positions[:, 2].min()

qd.translate([0, 0, zmax_slab - zmin_qd + vacuum_gap])

# Step 4: Center QD in x–y plane if desired
slab_center = slab_supercell.get_center_of_mass()[:2]
qd_center = qd.get_center_of_mass()[:2]
qd.translate([slab_center[0] - qd_center[0], slab_center[1] - qd_center[1], 0])

# Step 5: Combine the two
combined = slab_supercell + qd

# Step 6: Optional — add vacuum on top
combined.center(vacuum=10, axis=2)

# Step 7: Visualize
view(combined, viewer='x3d')

# Step 8: Save to file
# from ase.io import write
write("QD_on_slab.cif", combined)
